# Example Notebook

In [ ]:
from pynwb import NWBHDF5IO
import numpy as np
import datajoint as dj
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from bisect import bisect_left

dj_local_conf_path = "/Users/pauladkisson/Documents/CatalystNeuro/Spyglass/spyglass/dj_local_conf.json"
dj.config.load(dj_local_conf_path)  # load config for database connection info

# General Spyglass Imports
import spyglass.common as sgc  # this import connects to the database
import spyglass.data_import as sgi
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
import spyglass.lfp as sglfp

# Spike Sorting Imports
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
import spyglass.spikesorting.v1 as sgs
from spyglass.spikesorting.analysis.v1.group import SortedSpikesGroup
from spyglass.spikesorting.analysis.v1.group import UnitSelectionParams
from spyglass.spikesorting.analysis.v1.unit_annotation import UnitAnnotation
from tqdm import tqdm

# Custom Table Imports
sys.path.append(
    "/Users/pauladkisson/Documents/CatalystNeuro/DudchenkoConv/woodcode/moore_2025/spyglass_extensions"
)
from imported_pseudo_emg import ImportedPseudoEMG
from imported_histology_images import ImportedHistologyImages

In [ ]:
sgc.Session()

## Single-session Analysis

We will start by looking at a single example session

In [ ]:
nwb_file_name = get_nwb_copy_filename("H3022-210806.nwb")
sgc.Session() & {"nwb_file_name": nwb_file_name}

By looking at the subject table, we can see that our subject has the wild-type genotype. 

In [ ]:
sgc.Subject() & {"subject_id": "H3022"}

By looking at the TaskEpoch table, we can see the 4 sleep-wake epochs. 

In [ ]:
sgc.TaskEpoch & {"nwb_file_name": nwb_file_name}

This table references interval lists which contain the actual start time and stop time of each of these epochs.
Let's look at the first epoch. 

In [ ]:
(sgc.IntervalList & {"nwb_file_name": nwb_file_name, "interval_list_name": "01"})

In [ ]:
start_time, stop_time = (sgc.IntervalList & {"nwb_file_name": nwb_file_name, "interval_list_name": "01"}).fetch1("valid_times")[0]
print(f"Start time: {start_time}, Stop time: {stop_time}")

Now let's get the behavioral tracking data for this session. 

In [ ]:
sgc.PositionSource.SpatialSeries & {"nwb_file_name": nwb_file_name}

In [ ]:
tracking_df = (sgc.RawPosition & {"nwb_file_name": nwb_file_name, "interval_list_name": "pos 0 valid times"}).fetch1_dataframe()
tracking_df.head()

Let's plot just the behavioral tracking data for epoch 1

In [ ]:
tracking_df_epoch1 = tracking_df[(tracking_df.index >= start_time) & (tracking_df.index <= stop_time)]
tracking_df_epoch1

In [ ]:
plt.figure(figsize=(15, 10))
x = np.array(tracking_df_epoch1['x'])
y = np.array(tracking_df_epoch1['y'])
timestamps = np.array(tracking_df_epoch1.index)

sc = plt.scatter(x, y, c=timestamps, cmap='viridis', s=1)
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Position Trajectory')
plt.colorbar(sc, label='Time (s)')

Let's also plot the head direction

In [ ]:
head_direction = (sgc.RawCompassDirection & {"nwb_file_name": nwb_file_name, "interval_list_name": "compass 1 valid times"}).fetch_nwb()[0]['compass']
head_direction

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Interpretable parameters ---
plot_duration_s = 10.0   # How many seconds of data to animate (from start_time)
playback_speed = 1.0     # 1.0 = real time, 2.0 = 2x faster, 0.5 = half speed

# Restrict to the first `plot_duration_s` seconds of the epoch
window_start = start_time
window_stop = min(start_time + plot_duration_s, stop_time)

mask = (head_direction.timestamps >= window_start) & (head_direction.timestamps <= window_stop)
head_direction_timestamps_epoch1 = head_direction.timestamps[mask]
head_direction_data_epoch1 = head_direction.data[mask]

# Infer sampling rate from the data, then pick a frame step that yields ~30 fps display
sampling_dt = np.median(np.diff(head_direction_timestamps_epoch1))
sampling_rate = 1.0 / sampling_dt
target_display_fps = 30.0
frame_step = max(1, int(round(sampling_rate / target_display_fps)))

sampled_head_direction_data = head_direction_data_epoch1[::frame_step]
sampled_head_direction_timestamps = head_direction_timestamps_epoch1[::frame_step]

# Real-time playback: each displayed frame represents (frame_step * sampling_dt) seconds
# of data. To play that back in real time, the interval between frames (in ms) must match
# that duration; scale by 1/playback_speed to speed up or slow down.
seconds_per_frame = frame_step * sampling_dt
interval_ms = (seconds_per_frame / playback_speed) * 1000.0

figure = plt.figure(figsize=(6, 6))
polar_axis = figure.add_subplot(111, projection="polar")
polar_axis.set_theta_zero_location("E")
polar_axis.set_theta_direction(1)
polar_axis.set_rlim(0, 1.0)
polar_axis.set_rticks([])
polar_axis.set_title(f"Head Direction (first {plot_duration_s:.0f}s, {playback_speed}x speed)")

(direction_line,) = polar_axis.plot(
    [sampled_head_direction_data[0], sampled_head_direction_data[0]],
    [0, 1],
    color="tab:blue",
    linewidth=2,
)
(direction_point,) = polar_axis.plot(
    [sampled_head_direction_data[0]],
    [1],
    marker="o",
    color="tab:red",
)
time_label = polar_axis.text(
    0.02,
    1.025,
    f"time = {sampled_head_direction_timestamps[0]:.2f} s",
    transform=polar_axis.transAxes,
)

def update_frame(frame_index):
    angle_value = sampled_head_direction_data[frame_index]
    direction_line.set_data([angle_value, angle_value], [0, 1])
    direction_point.set_data([angle_value], [1])
    time_label.set_text(f"time = {sampled_head_direction_timestamps[frame_index]:.2f} s")
    return direction_line, direction_point, time_label

head_direction_animation = FuncAnimation(
    figure,
    update_frame,
    frames=len(sampled_head_direction_data),
    interval=interval_ms,
    blit=True,
)

gif_filename = "head_direction.gif"
gif_fps = 1000.0 / interval_ms
head_direction_animation.save(gif_filename, writer="pillow", fps=gif_fps)

plt.close(figure)
HTML(head_direction_animation.to_jshtml())

Now let's get the sleep data for this session

In [ ]:
sgc.IntervalList & {"nwb_file_name": nwb_file_name}

In [ ]:
sgc.Task()

In [ ]:
sgc.TaskEpoch & {"nwb_file_name": nwb_file_name}

In [ ]:
nrem_times = (sgc.IntervalList & {"nwb_file_name": nwb_file_name, "interval_list_name": "sleep_nrem"}).fetch1("valid_times")
rem_times = (sgc.IntervalList & {"nwb_file_name": nwb_file_name, "interval_list_name": "sleep_rem"}).fetch1("valid_times")
wake_times = (sgc.IntervalList & {"nwb_file_name": nwb_file_name, "interval_list_name": "sleep_wake"}).fetch1("valid_times")
epoch_times = []
epoch_task_names = []
for epoch in ["01", "02", "03", "04"]:
    start_time, stop_time = (sgc.IntervalList & {"nwb_file_name": nwb_file_name, "interval_list_name": epoch}).fetch1("valid_times")[0]
    epoch_times.append((start_time, stop_time))
    epoch_int = int(epoch)
    epoch_task_name = (sgc.TaskEpoch & {"nwb_file_name": nwb_file_name, "epoch": epoch_int}).fetch1("task_name")
    epoch_task_names.append(epoch_task_name)

Plot example

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(15, 6), sharex=True)

# First subplot: Sleep stages
for start, stop in nrem_times:
    ax[0].axvspan(start, stop, alpha=0.5, color='blue', label='NREM')

for start, stop in rem_times:
    ax[0].axvspan(start, stop, alpha=0.5, color='red', label='REM')

for start, stop in wake_times:
    ax[0].axvspan(start, stop, alpha=0.5, color='green', label='Wake')

# Remove duplicate labels in legend for first subplot
handles, labels = ax[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax[0].legend(by_label.values(), by_label.keys())
ax[0].set_ylabel('Sleep Stage')
ax[0].set_yticks([])
ax[0].set_title('Sleep Stages Over Time')

# Second subplot: Epochs
task_color_map = {}
color_palette = ['orange', 'purple', 'cyan', 'yellow']
color_idx = 0

for i, (start, stop) in enumerate(epoch_times):
    task_name = epoch_task_names[i]
    if task_name not in task_color_map:
        task_color_map[task_name] = color_palette[color_idx % len(color_palette)]
        color_idx += 1
    # Only add label for first occurrence of each task
    label = task_name if task_name not in [h.get_label() for h in ax[1].get_children() if hasattr(h, 'get_label')] else None
    ax[1].axvspan(start, stop, alpha=0.5, color=task_color_map[task_name], label=label)

ax[1].legend()
ax[1].set_ylabel('Epoch')
ax[1].set_yticks([])
ax[1].set_title('Epochs Over Time')

ax[1].set_xlabel('Time (s)')
plt.tight_layout()


Now let's get the ephys data for this session. 

In [ ]:
plotting_start_time = 600.0
plotting_stop_time = plotting_start_time + 5.0  # 5 second window
print(f"Plotting window: {plotting_start_time} to {plotting_stop_time} seconds")
electrical_series = (sgc.Raw & {"nwb_file_name": nwb_file_name}).fetch_nwb()[0]["raw"]
plotting_start_index = bisect_left(electrical_series.timestamps, plotting_start_time)
plotting_stop_index = bisect_left(electrical_series.timestamps, plotting_stop_time)
plotting_slice = slice(plotting_start_index, plotting_stop_index)
print(f"Plotting indices: {plotting_start_index} to {plotting_stop_index} (exclusive)")

lfp_electrical_series = (sglfp.ImportedLFP & {"nwb_file_name": nwb_file_name}).fetch_nwb()[0]["lfp"]
lfp_plotting_start_index = bisect_left(lfp_electrical_series.timestamps, plotting_start_time)
lfp_plotting_stop_index = bisect_left(lfp_electrical_series.timestamps, plotting_stop_time)
lfp_plotting_slice = slice(lfp_plotting_start_index, lfp_plotting_stop_index)
print(f"LFP Plotting indices: {lfp_plotting_start_index} to {lfp_plotting_stop_index} (exclusive)")

In [ ]:
sgc.Electrode()

Retrieve Ephys Data for first electrode

In [ ]:
# Ephys
electrical_series_data = electrical_series.data[plotting_slice, 0]
raw_to_uV = electrical_series.conversion * 1e6
electrical_series_in_uV = electrical_series_data * raw_to_uV
electrical_series_timestamps = np.asarray(electrical_series.timestamps[plotting_slice])
lfp_data = np.asarray(lfp_electrical_series.data[lfp_plotting_slice, 0])
lfp_conversion = lfp_electrical_series.conversion * 1e6
lfp_in_uV = lfp_data * lfp_conversion
lfp_timestamps = np.asarray(lfp_electrical_series.timestamps[lfp_plotting_slice])

Plot example

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
ax[0].plot(electrical_series_timestamps, electrical_series_in_uV)
ax[0].set_ylabel("Voltage (uV)")
ax[0].set_title("Ephys Data")

ax[1].plot(lfp_timestamps, lfp_in_uV, color='black')
ax[1].set_ylabel("Voltage (uV)")
ax[1].set_title("LFP Data")
ax[1].set_xlabel("Time (s)")

Let's also look at the sorted spikes

In [ ]:
SortedSpikesGroup()

In [ ]:
SpikeSortingOutput()

In [ ]:
dir(SortedSpikesGroup()

In [ ]:
sgs.ImportedSpikeSorting()

In [ ]:
(sgs.ImportedSpikeSorting.Annotations & {"nwb_file_name": nwb_file_name, "id": 0}).fetch1("annotations")

In [ ]:
group_key = {
        "nwb_file_name": nwb_file_name,
        "sorted_spikes_group_name": "all_units",
    }
group_key = (SortedSpikesGroup & group_key).fetch1("KEY")
spike_data, unit_ids = SortedSpikesGroup().fetch_spike_data(group_key, return_unit_ids=True)
spike_data

Now, let's see the histology images for this session.

In [ ]:
ImportedHistologyImages & {"nwb_file_name": nwb_file_name}

In [ ]:
histology_images = (ImportedHistologyImages & {"nwb_file_name": nwb_file_name}).fetch_nwb()
for img_dict in histology_images:
    name = img_dict["image_name"]
    image_data = img_dict["histology_image"].data[:]
    plt.figure()
    plt.imshow(image_data)
    plt.axis("off")
    plt.title(name)
    plt.show()